# Notebook 1: TruthfulQA — Data Loading, Exploration & Baseline Generation

**Project:** Detection of Hallucination in LLMs and Exploration of Mitigation Techniques  
**Task:** Open-Domain Question Answering  
**Benchmark:** TruthfulQA (Lin et al., 2022)  

This notebook covers:
1. Loading and exploring the TruthfulQA dataset
2. Understanding its structure and categories
3. Generating baseline answers from our target models
4. Computing initial truthfulness scores

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install -q datasets transformers accelerate bitsandbytes scipy sentencepiece protobuf
!pip install -q rouge-score bert-score pandas matplotlib seaborn

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Load TruthfulQA Dataset

In [ ]:
# Load TruthfulQA from HuggingFace
# The 'generation' split is what we need — it has questions with best/correct/incorrect answers
dataset = load_dataset("truthful_qa", "generation")

# Convert to pandas for easier exploration
df = pd.DataFrame(dataset["validation"])
print(f"Total questions: {len(df)}")
print(f"\nColumns: {list(df.columns)}")

In [ ]:
# Look at a few examples to understand the structure
for i in range(3):
    print(f"{'='*80}")
    print(f"Question {i+1}: {df.iloc[i]['question']}")
    print(f"\nCategory: {df.iloc[i]['category']}")
    print(f"\nBest answer: {df.iloc[i]['best_answer']}")
    print(f"\nCorrect answers: {df.iloc[i]['correct_answers']}")
    print(f"\nIncorrect answers: {df.iloc[i]['incorrect_answers']}")
    print()

### Understanding the dataset structure

Each entry in TruthfulQA contains:
- **question**: The question itself (designed to be tricky — triggers common misconceptions)
- **best_answer**: The single best truthful answer
- **correct_answers**: A list of all acceptable truthful answers
- **incorrect_answers**: A list of common but wrong answers (hallucinations we'd expect)
- **category**: What type of misconception the question targets
- **source**: Where the question comes from

## 3. Dataset Exploration

In [ ]:
# What categories of questions are there?
category_counts = df['category'].value_counts()
print(f"Number of categories: {len(category_counts)}")
print(f"\nCategory distribution:")
print(category_counts.to_string())

In [ ]:
# Visualize the category distribution
plt.figure(figsize=(14, 8))
category_counts.plot(kind='barh', color='steelblue')
plt.xlabel('Number of Questions')
plt.ylabel('Category')
plt.title('TruthfulQA: Distribution of Question Categories')
plt.tight_layout()
plt.savefig('category_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# How many correct vs incorrect reference answers per question?
df['n_correct'] = df['correct_answers'].apply(len)
df['n_incorrect'] = df['incorrect_answers'].apply(len)

print(f"Correct answers per question — mean: {df['n_correct'].mean():.1f}, "
      f"min: {df['n_correct'].min()}, max: {df['n_correct'].max()}")
print(f"Incorrect answers per question — mean: {df['n_incorrect'].mean():.1f}, "
      f"min: {df['n_incorrect'].min()}, max: {df['n_incorrect'].max()}")

In [ ]:
# Look at some tricky questions — these are designed to fool LLMs
print("Examples of tricky questions that commonly cause hallucination:\n")
sample_indices = [0, 10, 50, 100, 200, 300, 400, 500]
for idx in sample_indices:
    if idx < len(df):
        row = df.iloc[idx]
        print(f"Q: {row['question']}")
        print(f"   Best answer: {row['best_answer']}")
        print(f"   Common wrong answer: {row['incorrect_answers'][0]}")
        print()

## 4. Baseline Answer Generation

Now we generate answers from our three target models. We start with **Mistral-7B-Instruct** as it's the strongest open-source 7B model and loads well on Colab Pro.

We use 4-bit quantization (QLoRA-style) to fit 7B models into Colab GPU memory.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 4-bit quantization config to fit 7B models on Colab
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

def load_model(model_name):
    """Load a model with 4-bit quantization."""
    print(f"Loading {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    print(f"Loaded {model_name} successfully.")
    return model, tokenizer

In [ ]:
def generate_answer(model, tokenizer, question, max_new_tokens=128):
    """
    Generate an answer for a single question.
    Returns the generated text and token-level log probabilities (for detection later).
    """
    # Format as instruction
    prompt = f"Answer the following question in 1-2 sentences. Be concise and factual.\n\nQuestion: {question}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,           # Greedy decoding for baseline
            output_scores=True,         # We need these for entropy-based detection
            return_dict_in_generate=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Extract generated text (excluding the prompt)
    generated_ids = outputs.sequences[0][inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    # Extract log probabilities for each generated token
    log_probs = []
    for i, score in enumerate(outputs.scores):
        probs = torch.softmax(score[0], dim=-1)
        token_id = generated_ids[i]
        token_prob = probs[token_id].item()
        log_probs.append({
            'token': tokenizer.decode([token_id]),
            'prob': token_prob,
            'log_prob': np.log(token_prob + 1e-10),
            'entropy': -(probs * torch.log(probs + 1e-10)).sum().item()
        })

    return answer, log_probs

In [ ]:
def generate_multiple_samples(model, tokenizer, question, n_samples=5, max_new_tokens=128):
    """
    Generate multiple stochastic samples for the same question.
    Used for SelfCheckGPT consistency-based detection later.
    """
    prompt = f"Answer the following question in 1-2 sentences. Be concise and factual.\n\nQuestion: {question}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    samples = []
    for _ in range(n_samples):
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,          # Stochastic sampling
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.pad_token_id,
            )
        generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
        answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        samples.append(answer)

    return samples

In [ ]:
# ============================================================
# MODEL 1: Mistral-7B-Instruct
# ============================================================
model_name = "mistralai/Mistral-7B-Instruct-v0.2"
model, tokenizer = load_model(model_name)

In [ ]:
# Quick sanity check — generate answers for 5 questions
print("Sanity check — generating answers for first 5 questions:\n")
for i in range(5):
    question = df.iloc[i]['question']
    answer, log_probs = generate_answer(model, tokenizer, question)
    print(f"Q: {question}")
    print(f"A: {answer}")
    print(f"   Best answer: {df.iloc[i]['best_answer']}")
    avg_entropy = np.mean([lp['entropy'] for lp in log_probs])
    print(f"   Avg entropy: {avg_entropy:.3f}")
    print()

In [ ]:
# ============================================================
# Full generation run — this takes a while (~30-60 min per model)
# Generates greedy answers + 5 stochastic samples per question
# ============================================================

from tqdm import tqdm

results_mistral = []

for idx in tqdm(range(len(df)), desc="Generating (Mistral-7B)"):
    row = df.iloc[idx]
    question = row['question']

    # Greedy answer + log probs
    answer, log_probs = generate_answer(model, tokenizer, question)

    # Multiple stochastic samples (for SelfCheckGPT later)
    samples = generate_multiple_samples(model, tokenizer, question, n_samples=5)

    results_mistral.append({
        'question_idx': idx,
        'question': question,
        'category': row['category'],
        'best_answer': row['best_answer'],
        'correct_answers': row['correct_answers'],
        'incorrect_answers': row['incorrect_answers'],
        'generated_answer': answer,
        'log_probs': log_probs,
        'stochastic_samples': samples,
    })

# Save results
import pickle
with open('results_mistral7b.pkl', 'wb') as f:
    pickle.dump(results_mistral, f)

print(f"\nGenerated answers for {len(results_mistral)} questions.")

In [ ]:
# Free GPU memory before loading next model
del model
del tokenizer
torch.cuda.empty_cache()
print("GPU memory cleared.")

In [ ]:
# ============================================================
# MODEL 2: LLaMA-2-7B-Chat
# NOTE: You may need to accept the license on HuggingFace first:
# https://huggingface.co/meta-llama/Llama-2-7b-chat-hf
# Then run: huggingface-cli login
# ============================================================
model_name = "meta-llama/Llama-2-7b-chat-hf"
model, tokenizer = load_model(model_name)

results_llama = []
for idx in tqdm(range(len(df)), desc="Generating (LLaMA-2-7B)"):
    row = df.iloc[idx]
    question = row['question']
    answer, log_probs = generate_answer(model, tokenizer, question)
    samples = generate_multiple_samples(model, tokenizer, question, n_samples=5)

    results_llama.append({
        'question_idx': idx,
        'question': question,
        'category': row['category'],
        'best_answer': row['best_answer'],
        'correct_answers': row['correct_answers'],
        'incorrect_answers': row['incorrect_answers'],
        'generated_answer': answer,
        'log_probs': log_probs,
        'stochastic_samples': samples,
    })

with open('results_llama2_7b.pkl', 'wb') as f:
    pickle.dump(results_llama, f)

print(f"Generated answers for {len(results_llama)} questions.")

del model, tokenizer
torch.cuda.empty_cache()

In [ ]:
# ============================================================
# MODEL 3: Phi-2 (2.7B — smaller baseline, fits easily on free Colab)
# ============================================================
model_name = "microsoft/phi-2"
model, tokenizer = load_model(model_name)

results_phi = []
for idx in tqdm(range(len(df)), desc="Generating (Phi-2)"):
    row = df.iloc[idx]
    question = row['question']
    answer, log_probs = generate_answer(model, tokenizer, question)
    samples = generate_multiple_samples(model, tokenizer, question, n_samples=5)

    results_phi.append({
        'question_idx': idx,
        'question': question,
        'category': row['category'],
        'best_answer': row['best_answer'],
        'correct_answers': row['correct_answers'],
        'incorrect_answers': row['incorrect_answers'],
        'generated_answer': answer,
        'log_probs': log_probs,
        'stochastic_samples': samples,
    })

with open('results_phi2.pkl', 'wb') as f:
    pickle.dump(results_phi, f)

print(f"Generated answers for {len(results_phi)} questions.")

del model, tokenizer
torch.cuda.empty_cache()

## 5. Baseline Truthfulness Evaluation

TruthfulQA evaluation checks if the model's answer is:
- **Truthful**: Does the answer avoid asserting false information? (Compared against correct/incorrect reference answers)
- **Informative**: Does the answer actually provide a substantive response? (Not just "I don't know")

We use a simple but effective heuristic approach: compute ROUGE-L and BERTScore similarity between the generated answer and both correct and incorrect reference answers. If it's more similar to correct answers, we label it truthful.

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

def compute_similarity_scores(generated, references):
    """Compute max ROUGE-L F1 between generated answer and a list of references."""
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    max_rouge = 0
    for ref in references:
        score = scorer.score(ref, generated)
        max_rouge = max(max_rouge, score['rougeL'].fmeasure)
    return max_rouge


def evaluate_truthfulness(results):
    """
    For each question, check if the generated answer is more similar
    to the correct answers or the incorrect answers.
    """
    evaluations = []

    for r in tqdm(results, desc="Evaluating truthfulness"):
        generated = r['generated_answer']

        # Similarity to correct answers
        correct_sim = compute_similarity_scores(generated, r['correct_answers'])

        # Similarity to incorrect answers
        incorrect_sim = compute_similarity_scores(generated, r['incorrect_answers'])

        # Simple classification: truthful if more similar to correct answers
        is_truthful = correct_sim > incorrect_sim

        # Check if informative (not just "I don't know" or empty)
        refusal_phrases = ["i don't know", "i'm not sure", "i cannot", "no answer", "unclear"]
        is_informative = not any(phrase in generated.lower() for phrase in refusal_phrases)

        evaluations.append({
            'question_idx': r['question_idx'],
            'question': r['question'],
            'category': r['category'],
            'generated_answer': generated,
            'best_answer': r['best_answer'],
            'correct_similarity': correct_sim,
            'incorrect_similarity': incorrect_sim,
            'is_truthful': is_truthful,
            'is_informative': is_informative,
            'is_truthful_and_informative': is_truthful and is_informative,
        })

    return pd.DataFrame(evaluations)

In [ ]:
# Load saved results and evaluate
import pickle

all_results = {}
model_files = {
    'Mistral-7B': 'results_mistral7b.pkl',
    'LLaMA-2-7B': 'results_llama2_7b.pkl',
    'Phi-2': 'results_phi2.pkl',
}

eval_dfs = {}
for model_name, filepath in model_files.items():
    try:
        with open(filepath, 'rb') as f:
            results = pickle.load(f)
        all_results[model_name] = results
        eval_dfs[model_name] = evaluate_truthfulness(results)
        print(f"{model_name}: loaded {len(results)} results")
    except FileNotFoundError:
        print(f"{model_name}: results file not found (run generation first)")

In [ ]:
# Print baseline truthfulness scores
print("=" * 60)
print("BASELINE TRUTHFULNESS SCORES")
print("=" * 60)

summary_rows = []
for model_name, eval_df in eval_dfs.items():
    truthful_pct = eval_df['is_truthful'].mean() * 100
    informative_pct = eval_df['is_informative'].mean() * 100
    both_pct = eval_df['is_truthful_and_informative'].mean() * 100

    print(f"\n{model_name}:")
    print(f"  Truthful:               {truthful_pct:.1f}%")
    print(f"  Informative:            {informative_pct:.1f}%")
    print(f"  Truthful + Informative: {both_pct:.1f}%")

    summary_rows.append({
        'Model': model_name,
        'Truthful (%)': round(truthful_pct, 1),
        'Informative (%)': round(informative_pct, 1),
        'Truthful + Informative (%)': round(both_pct, 1),
    })

summary_df = pd.DataFrame(summary_rows)
print("\n")
print(summary_df.to_string(index=False))

In [ ]:
# Visualize truthfulness by category (for one model)
if eval_dfs:
    model_to_plot = list(eval_dfs.keys())[0]
    edf = eval_dfs[model_to_plot]

    category_truth = edf.groupby('category')['is_truthful'].mean().sort_values()

    plt.figure(figsize=(14, 10))
    colors = ['#d9534f' if v < 0.5 else '#5cb85c' for v in category_truth.values]
    category_truth.plot(kind='barh', color=colors)
    plt.axvline(x=0.5, color='gray', linestyle='--', alpha=0.7)
    plt.xlabel('Truthfulness Rate')
    plt.ylabel('Category')
    plt.title(f'{model_to_plot}: Truthfulness by Question Category')
    plt.tight_layout()
    plt.savefig('truthfulness_by_category.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Look at some hallucinated examples
if eval_dfs:
    model_to_inspect = list(eval_dfs.keys())[0]
    edf = eval_dfs[model_to_inspect]

    hallucinated = edf[~edf['is_truthful']].head(10)

    print(f"Examples of hallucinated answers from {model_to_inspect}:\n")
    for _, row in hallucinated.iterrows():
        print(f"Q: {row['question']}")
        print(f"Model said: {row['generated_answer']}")
        print(f"Best answer: {row['best_answer']}")
        print(f"Category: {row['category']}")
        print()

## 6. Save Everything for Next Notebook

The next notebook (Notebook 2) will implement the three hallucination detection methods using the generated answers and log probabilities saved here.

In [ ]:
# Save evaluation DataFrames
for model_name, edf in eval_dfs.items():
    safe_name = model_name.lower().replace('-', '_').replace(' ', '_')
    edf.to_csv(f'eval_{safe_name}.csv', index=False)
    print(f"Saved eval_{safe_name}.csv")

# Save summary
summary_df.to_csv('baseline_summary.csv', index=False)
print("\nSaved baseline_summary.csv")
print("\nAll pickle files with raw results (including log_probs and samples) are also saved.")
print("Ready for Notebook 2: Hallucination Detection Methods.")